# Phase 4: Portfolio Construction & Risk Management  
## Notebook 04_05 — PCA and Spectral Risk Analysis

### Research question

How can PCA and eigenvalue-spectrum analysis reveal hidden risk modes in a multi-asset portfolio, and how can those modes be used for covariance denoising, stress testing, and risk attribution?

This notebook follows:

```text
04_01_multi_asset_return_engine
04_02_capm_and_performance_attribution
04_03_volatility_targeting_and_position_sizing
04_04_mean_variance_optimization_ledoit_wolf
```

The previous notebook built covariance estimates for optimisation. This notebook opens the covariance matrix and asks:

> What are the dominant risk directions inside the covariance/correlation matrix?

It covers:

1. covariance and correlation eigen-decomposition;
2. PCA as risk-mode extraction;
3. eigenvalue spectrum;
4. explained variance;
5. market mode interpretation;
6. sector/asset-class modes;
7. principal component score time series;
8. principal portfolios;
9. portfolio exposure to PCs;
10. risk contribution by eigenmode;
11. Marčenko-Pastur-style noise-band intuition;
12. eigenvalue clipping / covariance denoising;
13. PCA stress scenarios;
14. rolling PCA stability;
15. regime-dependent spectral risk;
16. audit checks;
17. saved outputs and manifest.

The key idea:

> PCA turns a noisy covariance matrix into interpretable orthogonal risk modes. The largest eigenvectors often reveal market, sector, duration, commodity, or crisis exposures that are hidden in asset-by-asset views.

## 1. PCA for portfolio risk

For return matrix $R$, covariance matrix:

$$
\Sigma = Cov(R)
$$

Eigen-decomposition:

$$
\Sigma = Q\Lambda Q^\top
$$

where:

- $Q$ contains orthonormal eigenvectors;
- $\Lambda$ contains eigenvalues;
- each eigenvector is a principal risk direction;
- each eigenvalue is variance along that direction.

For portfolio weights $w$:

$$
\sigma_p^2 = w^\top\Sigma w
$$

Using the eigen-decomposition:

$$
\sigma_p^2 = w^\top Q\Lambda Q^\top w = \sum_k \lambda_k (q_k^\top w)^2
$$

This decomposes portfolio variance into contributions from orthogonal risk modes.

## 2. Correlation PCA versus covariance PCA

### Covariance PCA

Uses $\Sigma$.

Large-volatility assets can dominate the first components.

Useful when actual dollar risk matters.

### Correlation PCA

Uses correlation matrix $C$.

Each asset is standardised to unit volatility.

Useful for understanding common co-movement structure without one high-volatility asset dominating.

This notebook computes both, but emphasises correlation PCA for interpretation and covariance PCA for risk attribution.

## 3. Spectral risk analysis

Here, “spectral risk” refers to the eigen-spectrum of the covariance/correlation matrix:

$$
\lambda_1 \ge \lambda_2 \ge \cdots \ge \lambda_N
$$

A concentrated spectrum means a few common modes dominate risk.

A flatter spectrum means risk is more diversified across independent directions.

Diagnostics:

- largest eigenvalue share;
- cumulative explained variance;
- effective number of eigenmodes;
- eigenvector loadings;
- rolling stability of eigenvectors;
- spectrum shifts during stress regimes.

This is different from coherent “spectral risk measures” in the Acerbi sense; the focus here is covariance eigen-spectrum risk.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class PCASpectralRiskConfig:
    n_days: int = 2_200
    annualisation: int = 252
    rolling_window: int = 252
    rolling_step: int = 21
    n_components_to_show: int = 5
    stress_sigma: float = 2.5
    seed: int = 42


config = PCASpectralRiskConfig()
config

## 5. Synthetic multi-asset return universe

We simulate a portfolio universe with hidden factors:

1. global market;
2. equity regional factor;
3. rates factor;
4. commodity factor;
5. crypto/high-beta factor;
6. trend/alternative factor;
7. stress regime.

PCA should recover dominant modes similar to these hidden structures.

In [ ]:
def simulate_pca_risk_universe(config: PCASpectralRiskConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(config.seed)

    dates = pd.bdate_range("2015-01-01", periods=config.n_days)

    assets = [
        "US_EQ", "EU_EQ", "JP_EQ", "EM_EQ",
        "US_BOND", "EU_BOND",
        "GOLD", "OIL", "COPPER",
        "FX_CARRY", "TREND_FOLLOW",
        "REIT", "BTC_PROXY"
    ]

    asset_class = {
        "US_EQ": "equity",
        "EU_EQ": "equity",
        "JP_EQ": "equity",
        "EM_EQ": "equity",
        "US_BOND": "bond",
        "EU_BOND": "bond",
        "GOLD": "commodity",
        "OIL": "commodity",
        "COPPER": "commodity",
        "FX_CARRY": "fx",
        "TREND_FOLLOW": "alternative",
        "REIT": "real_asset",
        "BTC_PROXY": "crypto",
    }

    factor_names = [
        "global_market",
        "equity_region",
        "rates",
        "commodity",
        "crypto_beta",
        "trend_alt",
    ]

    factor_vol_ann = np.array([0.13, 0.08, 0.06, 0.11, 0.28, 0.09])
    factor_vol_daily = factor_vol_ann / np.sqrt(config.annualisation)

    factor_corr = np.array([
        [ 1.00,  0.45, -0.25,  0.25,  0.35, -0.05],
        [ 0.45,  1.00, -0.10,  0.10,  0.20, -0.05],
        [-0.25, -0.10,  1.00,  0.10, -0.15,  0.10],
        [ 0.25,  0.10,  0.10,  1.00,  0.15,  0.05],
        [ 0.35,  0.20, -0.15,  0.15,  1.00,  0.00],
        [-0.05, -0.05,  0.10,  0.05,  0.00,  1.00],
    ])

    factor_cov = np.outer(factor_vol_daily, factor_vol_daily) * factor_corr

    loadings = pd.DataFrame(0.0, index=assets, columns=factor_names)

    loadings.loc[["US_EQ", "EU_EQ", "JP_EQ", "EM_EQ", "REIT"], "global_market"] = [1.00, 0.95, 0.80, 1.15, 0.75]
    loadings.loc[["US_EQ", "EU_EQ", "JP_EQ", "EM_EQ"], "equity_region"] = [0.25, 0.35, 0.30, 0.55]
    loadings.loc[["US_BOND", "EU_BOND"], "rates"] = [1.00, 0.90]
    loadings.loc[["GOLD", "OIL", "COPPER"], "commodity"] = [0.45, 1.10, 0.95]
    loadings.loc["BTC_PROXY", "crypto_beta"] = 1.00
    loadings.loc["FX_CARRY", "global_market"] = 0.25
    loadings.loc["FX_CARRY", "trend_alt"] = 0.20
    loadings.loc["TREND_FOLLOW", "trend_alt"] = 1.00
    loadings.loc["TREND_FOLLOW", "global_market"] = -0.20
    loadings.loc["GOLD", "rates"] = 0.20
    loadings.loc["REIT", "rates"] = -0.25

    idio_vol_ann = pd.Series({
        "US_EQ": 0.07,
        "EU_EQ": 0.08,
        "JP_EQ": 0.09,
        "EM_EQ": 0.12,
        "US_BOND": 0.025,
        "EU_BOND": 0.030,
        "GOLD": 0.12,
        "OIL": 0.20,
        "COPPER": 0.18,
        "FX_CARRY": 0.07,
        "TREND_FOLLOW": 0.08,
        "REIT": 0.10,
        "BTC_PROXY": 0.45,
    })

    regime = np.zeros(config.n_days, dtype=int)
    transition = np.array([[0.985, 0.015], [0.050, 0.950]])

    factor_returns = np.zeros((config.n_days, len(factor_names)))
    asset_returns = np.zeros((config.n_days, len(assets)))

    annual_means = pd.Series({
        "US_EQ": 0.07,
        "EU_EQ": 0.06,
        "JP_EQ": 0.045,
        "EM_EQ": 0.08,
        "US_BOND": 0.025,
        "EU_BOND": 0.020,
        "GOLD": 0.035,
        "OIL": 0.045,
        "COPPER": 0.040,
        "FX_CARRY": 0.030,
        "TREND_FOLLOW": 0.050,
        "REIT": 0.060,
        "BTC_PROXY": 0.120,
    })

    for t in range(config.n_days):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_multiplier = 1.0 if regime[t] == 0 else 2.4
        factor_cov_t = factor_cov * vol_multiplier**2

        f = rng.multivariate_normal(np.zeros(len(factor_names)), factor_cov_t)

        # Stress impulse.
        if rng.uniform() < 0.007:
            f[0] -= rng.uniform(0.025, 0.080)
            f[2] += rng.uniform(0.003, 0.020)
            f[3] += rng.normal(0.000, 0.035)
            f[4] -= rng.uniform(0.050, 0.160)
            f[5] += rng.uniform(0.005, 0.035)

        eps = rng.standard_t(df=7, size=len(assets)) * np.sqrt((7 - 2) / 7)
        eps = eps * (idio_vol_ann.values / np.sqrt(config.annualisation))

        mu_daily = annual_means.values / config.annualisation
        asset_returns[t] = mu_daily + loadings.to_numpy() @ f + eps
        factor_returns[t] = f

    returns_df = pd.DataFrame(asset_returns, columns=assets)
    returns_df.insert(0, "date", dates)
    returns_df["regime"] = regime

    factors_df = pd.DataFrame(factor_returns, columns=factor_names)
    factors_df.insert(0, "date", dates)
    factors_df["regime"] = regime

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "idio_vol_ann": [idio_vol_ann[a] for a in assets],
        "annual_mean_assumption": [annual_means[a] for a in assets],
    })

    return returns_df, factors_df, metadata


returns_df, hidden_factors, asset_metadata = simulate_pca_risk_universe(config)
asset_cols = asset_metadata["asset"].tolist()
returns = returns_df.set_index("date")[asset_cols]

returns_df.head(), hidden_factors.head(), asset_metadata

In [ ]:
plt.figure(figsize=(12, 6))
for asset in asset_cols:
    plt.plot(returns.index, (1 + returns[asset]).cumprod(), label=asset, alpha=0.75)
plt.title("Synthetic Asset Growth of 1")
plt.xlabel("Date")
plt.ylabel("Growth")
plt.legend(ncol=3)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(returns_df["date"], returns_df["regime"], drawstyle="steps-post")
plt.title("Volatility Regime")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 6. Return diagnostics

Before PCA, inspect asset volatility and correlation.

High-volatility assets can dominate covariance PCA.

In [ ]:
return_summary = returns.agg(["mean", "std", "min", "max"]).T.rename(columns={
    "mean": "daily_mean",
    "std": "daily_vol",
    "min": "min_daily_return",
    "max": "max_daily_return",
})

return_summary["annualised_mean"] = return_summary["daily_mean"] * config.annualisation
return_summary["annualised_vol"] = return_summary["daily_vol"] * np.sqrt(config.annualisation)
return_summary["asset_class"] = return_summary.index.map(asset_metadata.set_index("asset")["asset_class"])

return_summary.sort_values("annualised_vol", ascending=False)

In [ ]:
corr = returns.corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr.values, vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(asset_cols)), asset_cols, rotation=90)
plt.yticks(range(len(asset_cols)), asset_cols)
plt.title("Asset Return Correlation Matrix")
plt.tight_layout()
plt.show()

## 7. PCA utilities

For a symmetric covariance or correlation matrix:

$$
A = Q\Lambda Q^\top
$$

We sort eigenvalues descending.

Explained variance ratio:

$$
EVR_k = \frac{\lambda_k}{\sum_j \lambda_j}
$$

Cumulative explained variance:

$$
CEV_K = \sum_{k=1}^{K}EVR_k
$$

In [ ]:
def eigen_decomposition_symmetric(matrix, labels):
    eigvals, eigvecs = np.linalg.eigh(matrix)

    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    # Fix arbitrary sign so the largest absolute loading is positive.
    for j in range(eigvecs.shape[1]):
        max_idx = np.argmax(np.abs(eigvecs[:, j]))
        if eigvecs[max_idx, j] < 0:
            eigvecs[:, j] *= -1

    explained = eigvals / eigvals.sum()
    cumulative = np.cumsum(explained)

    eig_table = pd.DataFrame({
        "component": np.arange(1, len(eigvals) + 1),
        "eigenvalue": eigvals,
        "explained_variance_ratio": explained,
        "cumulative_explained_variance": cumulative
    })

    loadings = pd.DataFrame(
        eigvecs,
        index=labels,
        columns=[f"PC{k}" for k in range(1, len(eigvals) + 1)]
    )

    return eig_table, loadings


def effective_number_of_components(eigenvalues):
    p = eigenvalues / np.sum(eigenvalues)
    return float(1.0 / np.sum(p**2))


cov_daily = returns.cov()
cov_ann = cov_daily * config.annualisation
corr_matrix = returns.corr()

eig_cov, load_cov = eigen_decomposition_symmetric(cov_ann.to_numpy(), asset_cols)
eig_corr, load_corr = eigen_decomposition_symmetric(corr_matrix.to_numpy(), asset_cols)

pd.Series({
    "effective_components_covariance": effective_number_of_components(eig_cov["eigenvalue"]),
    "effective_components_correlation": effective_number_of_components(eig_corr["eigenvalue"]),
    "first_pc_corr_explained_variance": eig_corr.loc[0, "explained_variance_ratio"],
})

## 8. Eigenvalue spectrum

The eigenvalue spectrum shows whether risk is concentrated in a few common modes.

A large first eigenvalue often indicates a broad market mode.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(eig_corr["component"], eig_corr["eigenvalue"], marker="o", label="correlation PCA")
ax.axhline(1.0, linestyle="--", label="average eigenvalue = 1")
ax.set_title("Correlation Matrix Eigenvalue Spectrum")
ax.set_xlabel("Component")
ax.set_ylabel("Eigenvalue")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(eig_corr["component"], eig_corr["explained_variance_ratio"])
ax.plot(eig_corr["component"], eig_corr["cumulative_explained_variance"], marker="o", label="cumulative")
ax.set_title("Explained Variance by Principal Component")
ax.set_xlabel("Component")
ax.set_ylabel("Explained variance ratio")
ax.legend()
plt.show()

eig_corr.head(10)

## 9. Principal component loadings

Correlation PCA loadings are easier to interpret because assets are standardised.

A first PC with same-sign loadings across risky assets is often the market mode.

Later PCs may separate:

- equities versus bonds;
- commodities versus financial assets;
- crypto/high beta;
- trend-following diversifier.

In [ ]:
load_corr.iloc[:, :config.n_components_to_show]

In [ ]:
for pc in [f"PC{i}" for i in range(1, config.n_components_to_show + 1)]:
    plt.figure(figsize=(10, 5))
    s = load_corr[pc].sort_values()
    plt.barh(s.index, s.values)
    plt.axvline(0, linestyle="--")
    plt.title(f"Correlation PCA Loadings: {pc}")
    plt.xlabel("Loading")
    plt.ylabel("Asset")
    plt.show()

## 10. PC scores

For standardised returns $Z_t$:

$$
score_{t,k}=Z_t q_k
$$

PC score time series show when each risk mode is active.

They are orthogonal by construction over the sample used for PCA.

In [ ]:
def standardize_returns(returns):
    return (returns - returns.mean()) / returns.std(ddof=1)


Z = standardize_returns(returns).fillna(0.0)
pc_scores = pd.DataFrame(
    Z.to_numpy() @ load_corr.to_numpy(),
    index=returns.index,
    columns=load_corr.columns
)

pc_scores.head()

In [ ]:
plt.figure(figsize=(12, 6))
for pc in ["PC1", "PC2", "PC3", "PC4"]:
    plt.plot(pc_scores.index, pc_scores[pc].rolling(21).mean(), label=f"{pc} 21d avg")
plt.axhline(0, linestyle="--")
plt.title("Principal Component Scores")
plt.xlabel("Date")
plt.ylabel("Score")
plt.legend()
plt.show()

## 11. PC scores versus hidden factors

Because this is synthetic, we know hidden factors.

We compare PC scores with hidden factor returns.

This helps interpret which economic risks each PC is capturing.

In [ ]:
hidden = hidden_factors.set_index("date").drop(columns=["regime"])
pc_factor_corr = pd.concat([pc_scores.iloc[:, :6], hidden], axis=1).corr().loc[pc_scores.columns[:6], hidden.columns]

pc_factor_corr

In [ ]:
plt.figure(figsize=(9, 6))
plt.imshow(pc_factor_corr.values, vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(hidden.columns)), hidden.columns, rotation=45, ha="right")
plt.yticks(range(6), pc_scores.columns[:6])
plt.title("PC Score Correlation with Hidden Factors")
plt.tight_layout()
plt.show()

## 12. Principal portfolios

Eigenvectors can be interpreted as principal portfolios.

For a component $q_k$, the principal portfolio holds weights proportional to $q_k$.

For display, we normalise each vector so gross exposure sums to 1:

$$
w^{PC_k} = \frac{q_k}{\sum_i |q_{k,i}|}
$$

These are not trading recommendations; they are risk-mode portfolios.

In [ ]:
def principal_portfolio_weights(loadings, n_components):
    cols = [f"PC{i}" for i in range(1, n_components + 1)]
    out = loadings[cols].copy()

    for col in cols:
        gross = out[col].abs().sum()
        out[col] = out[col] / gross if gross > 0 else out[col]

    return out


principal_portfolios = principal_portfolio_weights(load_corr, config.n_components_to_show)
principal_portfolios

In [ ]:
for pc in principal_portfolios.columns:
    plt.figure(figsize=(10, 5))
    s = principal_portfolios[pc].sort_values()
    plt.barh(s.index, s.values)
    plt.axvline(0, linestyle="--")
    plt.title(f"Principal Portfolio Weights: {pc}")
    plt.xlabel("Gross-normalised weight")
    plt.ylabel("Asset")
    plt.show()

## 13. Portfolio PC exposure

For portfolio weights $w$, covariance PCA gives risk-mode exposure:

$$
e_k = q_k^\top w
$$

Variance contribution from mode $k$:

$$
VC_k = \lambda_k e_k^2
$$

Fractional contribution:

$$
FVC_k = \frac{\lambda_k e_k^2}{w^\top\Sigma w}
$$

This tells us whether portfolio risk is dominated by one or two hidden modes.

In [ ]:
def example_portfolios(asset_cols):
    n = len(asset_cols)
    equal = pd.Series(1.0 / n, index=asset_cols)

    # Risky portfolio tilted to equities, commodities, crypto.
    risky = pd.Series(0.0, index=asset_cols)
    for asset in ["US_EQ", "EU_EQ", "JP_EQ", "EM_EQ"]:
        risky[asset] = 0.12
    for asset in ["OIL", "COPPER", "BTC_PROXY", "REIT"]:
        risky[asset] = 0.10
    risky["GOLD"] = 0.06
    risky["TREND_FOLLOW"] = 0.06
    risky["US_BOND"] = 0.03
    risky["EU_BOND"] = 0.03
    risky["FX_CARRY"] = 0.04
    risky = risky / risky.sum()

    # Defensive portfolio.
    defensive = pd.Series(0.0, index=asset_cols)
    defensive["US_BOND"] = 0.25
    defensive["EU_BOND"] = 0.20
    defensive["GOLD"] = 0.15
    defensive["TREND_FOLLOW"] = 0.15
    defensive["US_EQ"] = 0.08
    defensive["EU_EQ"] = 0.06
    defensive["JP_EQ"] = 0.04
    defensive["EM_EQ"] = 0.03
    defensive["REIT"] = 0.03
    defensive["FX_CARRY"] = 0.01
    defensive = defensive / defensive.sum()

    return pd.DataFrame({
        "equal_weight": equal,
        "risky_tilt": risky,
        "defensive_tilt": defensive,
    })


portfolio_weights = example_portfolios(asset_cols)
portfolio_weights

In [ ]:
def spectral_risk_decomposition(weights, eig_table, loadings):
    eigvals = eig_table["eigenvalue"].to_numpy()
    Q = loadings.to_numpy()

    rows = []

    for portfolio_name in weights.columns:
        w = weights[portfolio_name].to_numpy()
        exposures = Q.T @ w
        variance_contrib = eigvals * exposures**2
        total_var = variance_contrib.sum()

        for k, vc in enumerate(variance_contrib, start=1):
            rows.append({
                "portfolio": portfolio_name,
                "component": f"PC{k}",
                "eigenvalue": eigvals[k - 1],
                "pc_exposure": exposures[k - 1],
                "variance_contribution": vc,
                "variance_contribution_pct": vc / total_var if total_var > 0 else np.nan
            })

    return pd.DataFrame(rows)


spectral_decomp_cov = spectral_risk_decomposition(portfolio_weights, eig_cov, load_cov)
spectral_decomp_cov.head()

In [ ]:
top_decomp = spectral_decomp_cov[spectral_decomp_cov["component"].isin([f"PC{i}" for i in range(1, 7)])]

plt.figure(figsize=(12, 6))
for portfolio in portfolio_weights.columns:
    sub = top_decomp[top_decomp["portfolio"] == portfolio]
    plt.plot(sub["component"], sub["variance_contribution_pct"], marker="o", label=portfolio)
plt.title("Portfolio Variance Contribution by PCA Mode")
plt.xlabel("Component")
plt.ylabel("Variance contribution")
plt.legend()
plt.show()

## 14. Marčenko-Pastur-style noise band intuition

For $N$ assets and $T$ observations, if returns were pure noise with no true correlations, correlation eigenvalues approximately lie in:

$$
\lambda_{\pm} = \sigma^2 \Big( 1 \pm \sqrt{\frac{N}{T}} \Big)^2
$$

For a correlation matrix, $\sigma^2 \approx 1$.

Eigenvalues above $\lambda_+$ may indicate genuine common structure.

This is an intuition, not a formal proof in this finite synthetic example.

In [ ]:
def marcenko_pastur_bounds(n_assets, n_obs, sigma2=1.0):
    q = n_assets / n_obs
    lower = sigma2 * (1 - np.sqrt(q))**2
    upper = sigma2 * (1 + np.sqrt(q))**2
    return lower, upper


mp_lower, mp_upper = marcenko_pastur_bounds(len(asset_cols), len(returns), sigma2=1.0)

mp_report = pd.Series({
    "n_assets": len(asset_cols),
    "n_observations": len(returns),
    "mp_lower": mp_lower,
    "mp_upper": mp_upper,
    "n_corr_eigenvalues_above_mp_upper": int((eig_corr["eigenvalue"] > mp_upper).sum())
})

mp_report

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(eig_corr["component"], eig_corr["eigenvalue"], marker="o", label="correlation eigenvalues")
plt.axhline(mp_upper, linestyle="--", label="MP upper")
plt.axhline(mp_lower, linestyle="--", label="MP lower")
plt.axhline(1.0, linestyle=":", label="average eigenvalue")
plt.title("Correlation Spectrum vs MP Noise Band")
plt.xlabel("Component")
plt.ylabel("Eigenvalue")
plt.legend()
plt.show()

## 15. Eigenvalue clipping / covariance denoising

A simple denoising approach:

1. eigen-decompose correlation matrix;
2. keep large eigenvalues above MP upper bound;
3. replace noisy eigenvalues by their average;
4. reconstruct denoised correlation;
5. convert back to covariance using original volatilities.

This is a simplified spectral shrinkage method.

In [ ]:
def denoise_correlation_by_eigenvalue_clipping(corr_matrix, mp_upper):
    labels = corr_matrix.index.tolist()
    eig_table, loadings = eigen_decomposition_symmetric(corr_matrix.to_numpy(), labels)

    eigvals = eig_table["eigenvalue"].to_numpy()
    Q = loadings.to_numpy()

    signal_mask = eigvals > mp_upper
    clipped_eigs = eigvals.copy()

    if (~signal_mask).sum() > 0:
        clipped_eigs[~signal_mask] = eigvals[~signal_mask].mean()

    denoised = Q @ np.diag(clipped_eigs) @ Q.T

    # Renormalise diagonal to 1.
    diag = np.sqrt(np.diag(denoised))
    denoised = denoised / np.outer(diag, diag)
    denoised = np.clip(denoised, -1, 1)

    return pd.DataFrame(denoised, index=labels, columns=labels), eigvals, clipped_eigs


denoised_corr, raw_eigs, clipped_eigs = denoise_correlation_by_eigenvalue_clipping(corr_matrix, mp_upper)

vol_daily = returns.std(ddof=1)
denoised_cov_daily = denoised_corr.multiply(vol_daily, axis=0).multiply(vol_daily, axis=1)
denoised_cov_ann = denoised_cov_daily * config.annualisation

denoised_corr.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(raw_eigs, marker="o", label="raw eigenvalues")
plt.plot(clipped_eigs, marker="x", label="clipped eigenvalues")
plt.axhline(mp_upper, linestyle="--", label="MP upper")
plt.title("Eigenvalue Clipping")
plt.xlabel("Component index")
plt.ylabel("Eigenvalue")
plt.legend()
plt.show()

## 16. Sample versus denoised covariance diagnostics

Denoising should reduce noise and often improve conditioning.

It may also remove real weak signals if applied carelessly.

In [ ]:
def covariance_diagnostics(cov, label):
    eigvals = np.linalg.eigvalsh(cov)
    return {
        "matrix": label,
        "min_eigenvalue": float(eigvals.min()),
        "max_eigenvalue": float(eigvals.max()),
        "condition_number": float(eigvals.max() / max(eigvals.min(), 1e-12)),
        "trace": float(np.trace(cov)),
        "effective_components": effective_number_of_components(eigvals)
    }


cov_diag = pd.DataFrame([
    covariance_diagnostics(cov_ann.to_numpy(), "sample_covariance"),
    covariance_diagnostics(denoised_cov_ann.to_numpy(), "spectral_denoised_covariance"),
])

cov_diag

## 17. PCA stress scenarios

A PCA stress shock moves returns along a principal component:

$$
shock^{(k)} = -a \sqrt{\lambda_k} q_k
$$

For correlation PCA, this shock is in standardised-return space.

We convert it back to asset return units using asset volatilities.

This gives interpretable stress scenarios:

- market-mode shock;
- rates/equity rotation shock;
- commodity shock;
- crypto/high-beta shock.

In [ ]:
def pca_stress_scenarios(loadings, eig_table, asset_vol_daily, n_components, stress_sigma):
    rows = []

    for k in range(1, n_components + 1):
        pc = f"PC{k}"
        eig = eig_table.loc[eig_table["component"] == k, "eigenvalue"].iloc[0]
        q = loadings[pc]

        # Negative shock along PC direction.
        shock_standardized = -stress_sigma * np.sqrt(eig) * q
        shock_return = shock_standardized * asset_vol_daily

        for asset, value in shock_return.items():
            rows.append({
                "scenario": f"{pc}_negative_{stress_sigma:.1f}sigma",
                "component": pc,
                "asset": asset,
                "shock_return": value
            })

    return pd.DataFrame(rows)


stress_scenarios = pca_stress_scenarios(
    load_corr,
    eig_corr,
    returns.std(ddof=1),
    config.n_components_to_show,
    config.stress_sigma
)

stress_scenarios.head()

In [ ]:
for scenario, sub in stress_scenarios.groupby("scenario"):
    plt.figure(figsize=(10, 5))
    s = sub.set_index("asset")["shock_return"].sort_values()
    plt.barh(s.index, s.values)
    plt.axvline(0, linestyle="--")
    plt.title(f"PCA Stress Scenario: {scenario}")
    plt.xlabel("One-day return shock")
    plt.ylabel("Asset")
    plt.show()

## 18. Portfolio losses under PCA stress

For portfolio weights $w$, stress PnL is:

$$
Loss = -w^\top shock
$$

We compute stress return for the example portfolios.

In [ ]:
def portfolio_stress_table(stress_scenarios, portfolio_weights):
    rows = []

    for scenario, sub in stress_scenarios.groupby("scenario"):
        shock = sub.set_index("asset")["shock_return"].reindex(portfolio_weights.index)

        for portfolio in portfolio_weights.columns:
            ret = float(portfolio_weights[portfolio] @ shock)
            rows.append({
                "scenario": scenario,
                "portfolio": portfolio,
                "stress_return": ret,
                "stress_loss": -ret
            })

    return pd.DataFrame(rows)


portfolio_stress = portfolio_stress_table(stress_scenarios, portfolio_weights)

portfolio_stress.head()

In [ ]:
plt.figure(figsize=(12, 6))
for portfolio in portfolio_weights.columns:
    sub = portfolio_stress[portfolio_stress["portfolio"] == portfolio]
    plt.plot(sub["scenario"], sub["stress_loss"], marker="o", label=portfolio)
plt.xticks(rotation=45, ha="right")
plt.title("Portfolio Loss Under PCA Stress Scenarios")
plt.ylabel("Stress loss")
plt.legend()
plt.tight_layout()
plt.show()

## 19. Rolling PCA

Risk modes change.

We compute rolling PCA over trailing windows and track:

1. first eigenvalue share;
2. effective number of components;
3. first PC similarity to the full-sample first PC;
4. regime dependence.

In [ ]:
def rolling_pca_diagnostics(returns, config):
    rows = []
    full_pc1 = load_corr["PC1"].to_numpy()

    for start in range(0, len(returns) - config.rolling_window + 1, config.rolling_step):
        window_returns = returns.iloc[start:start + config.rolling_window]
        date = window_returns.index[-1]

        c = window_returns.corr().to_numpy()
        eig_table, loadings = eigen_decomposition_symmetric(c, asset_cols)

        pc1 = loadings["PC1"].to_numpy()
        similarity = abs(float(pc1 @ full_pc1))

        rows.append({
            "date": date,
            "first_eigenvalue": eig_table.loc[0, "eigenvalue"],
            "first_pc_explained_variance": eig_table.loc[0, "explained_variance_ratio"],
            "effective_components": effective_number_of_components(eig_table["eigenvalue"]),
            "pc1_similarity_to_full_sample": similarity,
            "regime_mean": returns_df.set_index("date").loc[window_returns.index, "regime"].mean()
        })

    return pd.DataFrame(rows)


rolling_pca = rolling_pca_diagnostics(returns, config)

rolling_pca.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(rolling_pca["date"], rolling_pca["first_pc_explained_variance"], marker="o", label="PC1 explained variance")
plt.plot(rolling_pca["date"], rolling_pca["pc1_similarity_to_full_sample"], marker="x", label="PC1 similarity")
plt.title("Rolling PCA Stability")
plt.xlabel("Date")
plt.ylabel("Metric")
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
plt.scatter(rolling_pca["regime_mean"], rolling_pca["first_pc_explained_variance"])
plt.title("Stress Regime Share vs First PC Explained Variance")
plt.xlabel("Mean regime in window")
plt.ylabel("PC1 explained variance")
plt.show()

## 20. Regime-dependent spectra

We compare PCA spectra in calm and stress regimes.

During stress, correlations often rise and the first eigenvalue can dominate more.

In [ ]:
regime_series = returns_df.set_index("date")["regime"].reindex(returns.index)
calm_returns = returns[regime_series == 0]
stress_returns = returns[regime_series == 1]

eig_calm, load_calm = eigen_decomposition_symmetric(calm_returns.corr().to_numpy(), asset_cols)
eig_stress, load_stress = eigen_decomposition_symmetric(stress_returns.corr().to_numpy(), asset_cols)

regime_spectrum = pd.DataFrame({
    "component": eig_calm["component"],
    "calm_eigenvalue": eig_calm["eigenvalue"],
    "stress_eigenvalue": eig_stress["eigenvalue"],
    "calm_evr": eig_calm["explained_variance_ratio"],
    "stress_evr": eig_stress["explained_variance_ratio"],
})

regime_spectrum.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(regime_spectrum["component"], regime_spectrum["calm_eigenvalue"], marker="o", label="calm")
plt.plot(regime_spectrum["component"], regime_spectrum["stress_eigenvalue"], marker="x", label="stress")
plt.title("Correlation Eigenvalue Spectrum by Regime")
plt.xlabel("Component")
plt.ylabel("Eigenvalue")
plt.legend()
plt.show()

## 21. Diversification diagnostics

Effective number of eigenmodes:

$$
N_{eff} = \frac{1}{\sum_k p_k^2}
$$

where:

$$
p_k = \frac{\lambda_k}{\sum_j \lambda_j}
$$

This measures diversification across risk modes.

A portfolio with many assets can still have few effective risk modes.

In [ ]:
diversification_report = pd.Series({
    "n_assets": len(asset_cols),
    "effective_components_full_corr": effective_number_of_components(eig_corr["eigenvalue"]),
    "effective_components_calm_corr": effective_number_of_components(eig_calm["eigenvalue"]),
    "effective_components_stress_corr": effective_number_of_components(eig_stress["eigenvalue"]),
    "pc1_explained_full": eig_corr.loc[0, "explained_variance_ratio"],
    "pc1_explained_calm": eig_calm.loc[0, "explained_variance_ratio"],
    "pc1_explained_stress": eig_stress.loc[0, "explained_variance_ratio"],
})

diversification_report

## 22. Audit checks

PCA analysis should pass numerical checks:

1. eigenvalues are non-negative;
2. eigenvectors are orthonormal;
3. explained variance sums to 1;
4. reconstructed matrix matches original;
5. denoised correlation has diagonal equal to 1.

In [ ]:
def pca_audit(matrix, eig_table, loadings, label):
    Q = loadings.to_numpy()
    eigvals = eig_table["eigenvalue"].to_numpy()

    reconstructed = Q @ np.diag(eigvals) @ Q.T

    rows = [
        {
            "check": f"{label}_eigenvalues_nonnegative",
            "value": float(eigvals.min()),
            "passed": bool(eigvals.min() > -1e-10)
        },
        {
            "check": f"{label}_orthonormal_eigenvectors",
            "value": float(np.max(np.abs(Q.T @ Q - np.eye(Q.shape[1])))),
            "passed": bool(np.max(np.abs(Q.T @ Q - np.eye(Q.shape[1]))) < 1e-8)
        },
        {
            "check": f"{label}_explained_variance_sums_to_one",
            "value": float(abs(eig_table["explained_variance_ratio"].sum() - 1.0)),
            "passed": bool(abs(eig_table["explained_variance_ratio"].sum() - 1.0) < 1e-10)
        },
        {
            "check": f"{label}_reconstruction_error",
            "value": float(np.max(np.abs(reconstructed - matrix))),
            "passed": bool(np.max(np.abs(reconstructed - matrix)) < 1e-8)
        }
    ]

    return pd.DataFrame(rows)


audit_corr = pca_audit(corr_matrix.to_numpy(), eig_corr, load_corr, "correlation_pca")
audit_cov = pca_audit(cov_ann.to_numpy(), eig_cov, load_cov, "covariance_pca")

denoised_diag_error = float(np.max(np.abs(np.diag(denoised_corr) - 1.0)))
audit_denoised = pd.DataFrame([{
    "check": "denoised_correlation_diagonal_equals_one",
    "value": denoised_diag_error,
    "passed": bool(denoised_diag_error < 1e-10)
}])

audit_report = pd.concat([audit_corr, audit_cov, audit_denoised], ignore_index=True)

audit_report

## 23. Practical checklist for PCA risk analysis

Before using PCA in a risk process, check:

1. **Covariance or correlation?**  
   Use covariance for portfolio risk, correlation for structure.

2. **Eigenvalue concentration**  
   Does one mode dominate?

3. **Eigenvector interpretability**  
   Do loadings map to economic risks?

4. **Rolling stability**  
   Are risk modes stable through time?

5. **Stress regime behaviour**  
   Does the first eigenvalue rise during stress?

6. **Noise band**  
   Are small eigenvalues likely noise?

7. **Denoising impact**  
   Does spectral clipping improve conditioning?

8. **Portfolio PC exposure**  
   Is portfolio variance dominated by one mode?

9. **Scenario relevance**  
   Do PC stress shocks make economic sense?

10. **Do not overinterpret**  
   PCA is statistical, not causal.

## 24. Saving outputs

The notebook saves:

1. synthetic returns;
2. hidden factors;
3. asset metadata;
4. return diagnostics;
5. covariance and correlation matrices;
6. eigenvalue tables;
7. loading tables;
8. PC score time series;
9. principal portfolio weights;
10. spectral risk decomposition;
11. MP noise-band report;
12. denoised covariance/correlation;
13. PCA stress scenarios;
14. portfolio stress results;
15. rolling PCA diagnostics;
16. regime spectra;
17. audit report;
18. manifest.

In [ ]:
output_dir = Path("data/processed/pca_spectral_risk_analysis_v1")
output_dir.mkdir(parents=True, exist_ok=True)

returns_path = output_dir / "synthetic_returns.csv"
hidden_factors_path = output_dir / "hidden_factors.csv"
metadata_path = output_dir / "asset_metadata.csv"
return_summary_path = output_dir / "return_summary.csv"
cov_path = output_dir / "covariance_annualized.csv"
corr_path = output_dir / "correlation_matrix.csv"
eig_cov_path = output_dir / "eigenvalues_covariance.csv"
eig_corr_path = output_dir / "eigenvalues_correlation.csv"
load_cov_path = output_dir / "loadings_covariance.csv"
load_corr_path = output_dir / "loadings_correlation.csv"
pc_scores_path = output_dir / "pc_scores.csv"
pc_factor_corr_path = output_dir / "pc_factor_correlation.csv"
principal_portfolios_path = output_dir / "principal_portfolios.csv"
portfolio_weights_path = output_dir / "example_portfolio_weights.csv"
spectral_decomp_path = output_dir / "spectral_risk_decomposition.csv"
mp_report_path = output_dir / "mp_noise_band_report.csv"
denoised_corr_path = output_dir / "denoised_correlation.csv"
denoised_cov_path = output_dir / "denoised_covariance_annualized.csv"
cov_diag_path = output_dir / "covariance_diagnostics.csv"
stress_scenarios_path = output_dir / "pca_stress_scenarios.csv"
portfolio_stress_path = output_dir / "portfolio_pca_stress_results.csv"
rolling_pca_path = output_dir / "rolling_pca_diagnostics.csv"
regime_spectrum_path = output_dir / "regime_spectrum.csv"
diversification_path = output_dir / "diversification_report.csv"
audit_path = output_dir / "audit_report.csv"
manifest_path = output_dir / "manifest.json"

returns_df.to_csv(returns_path, index=False)
hidden_factors.to_csv(hidden_factors_path, index=False)
asset_metadata.to_csv(metadata_path, index=False)
return_summary.to_csv(return_summary_path)
cov_ann.to_csv(cov_path)
corr_matrix.to_csv(corr_path)
eig_cov.to_csv(eig_cov_path, index=False)
eig_corr.to_csv(eig_corr_path, index=False)
load_cov.to_csv(load_cov_path)
load_corr.to_csv(load_corr_path)
pc_scores.to_csv(pc_scores_path)
pc_factor_corr.to_csv(pc_factor_corr_path)
principal_portfolios.to_csv(principal_portfolios_path)
portfolio_weights.to_csv(portfolio_weights_path)
spectral_decomp_cov.to_csv(spectral_decomp_path, index=False)
mp_report.to_frame("value").to_csv(mp_report_path)
denoised_corr.to_csv(denoised_corr_path)
denoised_cov_ann.to_csv(denoised_cov_path)
cov_diag.to_csv(cov_diag_path, index=False)
stress_scenarios.to_csv(stress_scenarios_path, index=False)
portfolio_stress.to_csv(portfolio_stress_path, index=False)
rolling_pca.to_csv(rolling_pca_path, index=False)
regime_spectrum.to_csv(regime_spectrum_path, index=False)
diversification_report.to_frame("value").to_csv(diversification_path)
audit_report.to_csv(audit_path, index=False)

manifest = {
    "dataset_name": "pca_spectral_risk_analysis_outputs",
    "schema_version": "pca_spectral_risk_analysis_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "asset_cols": asset_cols,
    "n_assets": len(asset_cols),
    "first_pc_explained_variance": float(eig_corr.loc[0, "explained_variance_ratio"]),
    "effective_components_full_corr": float(diversification_report["effective_components_full_corr"]),
    "mp_report": mp_report.to_dict(),
    "audit_passed": bool(audit_report["passed"].all()),
    "audit_report": audit_report.to_dict(orient="records"),
    "notes": [
        "Correlation PCA is used for structure; covariance PCA is used for portfolio variance decomposition.",
        "The first principal component often represents a broad market/common-risk mode.",
        "Marcenko-Pastur-style bounds are used as intuition for noisy eigenvalues, not formal proof.",
        "Eigenvalue clipping is implemented as a simple spectral denoising method.",
        "PCA stress scenarios shock the portfolio along principal risk directions.",
        "Rolling PCA diagnostics show whether the spectrum and PC1 are stable through time."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

returns_path, eig_corr_path, spectral_decomp_path, rolling_pca_path, manifest_path

## 25. Limitations

### 25.1 Synthetic data

The notebook uses synthetic returns with known hidden factors.

Real market data has structural breaks, missing data, roll effects, and changing correlations.

### 25.2 PCA is linear

PCA captures linear covariance structure.

It does not capture nonlinear dependence, tail dependence, or option convexity.

### 25.3 Eigenvectors are sign-arbitrary

A PC vector can be multiplied by $-1$ without changing the component.

Interpret signs consistently.

### 25.4 PCA is statistical, not causal

A component may look like a market mode, but PCA itself does not identify causal economic drivers.

### 25.5 Small eigenvalues are noisy

Interpreting small PCs is dangerous.

Many small eigenvectors are dominated by estimation error.

### 25.6 MP bounds are approximate

The Marčenko-Pastur bounds assume idealised random matrices.

Real returns violate those assumptions.

### 25.7 Denoising can remove weak real signals

Eigenvalue clipping is useful but can erase genuine structure.

## 26. Research frontier and extensions

Important extensions include:

1. **Random matrix theory covariance filtering**  
   More formal eigenvalue cleaning.

2. **Hierarchical PCA**  
   PCA within asset classes before global PCA.

3. **Sparse PCA**  
   More interpretable components.

4. **Dynamic PCA**  
   Time-varying eigenvectors and eigenvalues.

5. **Kernel PCA**  
   Nonlinear risk modes.

6. **Tail PCA / quantile PCA**  
   Components focused on downside or tail events.

7. **Robust covariance PCA**  
   Use robust covariance estimators before eigen-decomposition.

8. **PCA factor risk model**  
   Use leading PCs as statistical risk factors.

9. **Stress testing by eigenmodes**  
   Build scenario libraries from historical PC shocks.

10. **Chinese futures PCA risk**  
   Extract commodity-chain risk modes across metals, energy, agriculture, and night-session dynamics.

## 27. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_06_risk_parity_and_equal_risk_contribution**  
   Allocate by risk contribution rather than return forecasts.

2. **04_07_var_cvar_and_stress_testing**  
   Use PCA stress shocks in tail-risk analysis.

3. **04_08_factor_risk_model_construction**  
   Build statistical and fundamental risk models.

4. **04_09_correlation_breakdown_and_regime_risk**  
   Study correlation spikes during stress.

5. **05_02_performance_attribution_report_template**  
   Include PCA risk attribution in reporting.

6. **07_01_multi_asset_cta_research_pipeline**  
   Use spectral risk modes in a futures portfolio.

## 28. Summary

This notebook implemented PCA and spectral risk analysis.

It showed how to:

1. simulate multi-asset returns with hidden factors;
2. compute covariance and correlation matrices;
3. eigen-decompose covariance and correlation matrices;
4. inspect eigenvalue spectra and explained variance;
5. interpret principal component loadings;
6. compute PC score time series;
7. compare PCs to hidden factors;
8. build principal portfolios;
9. decompose portfolio risk by eigenmode;
10. apply Marčenko-Pastur-style noise-band intuition;
11. denoise correlation/covariance by eigenvalue clipping;
12. construct PCA stress scenarios;
13. compute portfolio stress losses;
14. monitor rolling PCA stability;
15. compare calm and stress spectra;
16. save outputs and audits.

The key computational takeaway:

> PCA decomposes covariance risk into orthogonal modes that can be inspected, denoised, monitored, and stressed.

The key financial takeaway:

> Diversification across assets is not the same as diversification across risk modes.

## 29. Further reading

- Jolliffe, *Principal Component Analysis.*
- Meucci, *Risk and Asset Allocation.*
- Ledoit and Wolf on covariance shrinkage and cleaning.
- Laloux, Cizeau, Bouchaud, and Potters on random matrix theory in finance.
- Bouchaud and Potters on financial correlations and eigenvalue cleaning.
- Literature on statistical risk models, PCA factor models, and covariance denoising.